In [1]:
import pandas as pd
from pathlib import Path

input_file_agg = "aggregated_results.csv"

df_agg = pd.read_csv(input_file_agg)
df_agg.head()

,file,scenario,git_info,max_iter,LLM,LLM_temp,NOFEEDBACK,GENCMD,RUNFUZZ,subject,commit,final_result,unintended_bug,time,iid,score,success
0,SUMMARY_jerryscript_BIC_GITDIFFONLY_ITER5_gpt-...,BIC,DIFFONLY,5,gpt-4o-2024-08-06,0.5,NaN,NaN,NaN,jerryscript,b7e3bae,R,False,100,5013,1,0
1,SUMMARY_jerryscript_BIC_GITDIFFONLY_ITER5_gpt-...,BIC,DIFFONLY,5,gpt-4o-2024-08-06,0.5,NaN,NaN,NaN,jerryscript,70e275e,D,False,97,5138,2,0
2,SUMMARY_jerryscript_BIC_GITDIFFONLY_ITER5_gpt-...,BIC,DIFFONLY,5,gpt-4o-2024-08-06,0.5,NaN,NaN,NaN,jerryscript,53b61c1,D,False,57,5117,2,0
3,SUMMARY_jerryscript_BIC_GITDIFFONLY_ITER5_gpt-...,BIC,DIFFONLY,5,gpt-4o-2024-08-06,0.5,NaN,NaN,NaN,jerryscript,70e275e,B,False,36,5153,3,1
4,SUMMARY_jerryscript_BIC_GITDIFFONLY_ITER5_gpt-...,BIC,DIFFONLY,5,gpt-4o-2024-08-06,0.5,NaN,NaN,NaN,jerryscript,b7e3bae,R,False,107,5013,1,0


In [2]:
# create targets mapping[subject][issue/BIC/BFC]
from collections import defaultdict
targets = defaultdict(dict)
for index, row in df_agg.iterrows():
    subject = row['subject']
    issue = row['iid']
    commit = row['commit']
    targets[subject]['issue'] = issue
    if row['scenario'] == 'FIX':
      targets[subject]['BFC'] = commit
    else:
      targets[subject]['BIC'] = commit

print(targets)


defaultdict(<class 'dict'>, {'jerryscript': {'issue': 5153, 'BIC': '70e275e', 'BFC': 'de51531'}, 'jq': {'issue': 49014, 'BIC': '4003202', 'BFC': '499c91b'}, 'libxml2': {'issue': 841, 'BIC': '9cfc723', 'BFC': 'dc6270d'}, 'micropython': {'issue': 17847, 'BIC': '7e14680', 'BFC': '0615d13'}, 'mujs': {'issue': 166, 'BIC': '3f71a1c', 'BFC': '8b5ba20'}, 'php-src': {'issue': 17463, 'BIC': '7904a08', 'BFC': 'e4473ab'}, 'poppler': {'issue': 1381, 'BIC': '245abad', 'BFC': '1be35ee'}, 'z3': {'issue': 7252, 'BIC': 'a7b1db6', 'BFC': 'a6b5027'}})


In [3]:
from difflib import SequenceMatcher
from termcolor import colored

def word_diff(a, b, latex=False):
  def preprocess(text):
    return text.replace('.', '').replace(',', '')

  matcher = SequenceMatcher(None, preprocess(a).split(), preprocess(b).split())
  diff = []
  for tag, i1, i2, j1, j2 in matcher.get_opcodes():
    if tag == 'replace':
      if latex:
        diff.append(f"\\textcolor{{red}}{{{' '.join(a.split()[i1:i2])}}}")
        diff.append(f"\\textcolor{{teal}}{{{' '.join(b.split()[j1:j2])}}}")
      else:
        diff.append(colored(' '.join(a.split()[i1:i2]), 'red'))
        diff.append(colored(' '.join(b.split()[j1:j2]), 'green'))
    elif tag == 'delete':
      if latex:
        diff.append(f"\\textcolor{{red}}{{{' '.join(a.split()[i1:i2])}}}")
      else:
        diff.append(colored(' '.join(a.split()[i1:i2]), 'red'))
    elif tag == 'insert':
      if latex:
        diff.append(f"\\textcolor{{teal}}{{{' '.join(b.split()[j1:j2])}}}")
      else:
        diff.append(colored(' '.join(b.split()[j1:j2]), 'green'))
    elif tag == 'equal':
      diff.append(' '.join(a.split()[i1:i2]))
  return ' '.join(diff)

# Example usage
msg_origin = 'Fix parsing of invalid asterix symbol after private field symbol.'
msg_enhanced = 'Fix parsing of invalid asterisk symbol after private field symbol, which previously caused a segmentation fault when an asterisk was used after an async private method declaration.'

# Output in terminal with colors
print(word_diff(msg_origin, msg_enhanced))

# Output in LaTeX format
print(word_diff(msg_origin, msg_enhanced, latex=True))

Fix parsing of invalid asterix asterisk symbol after private field symbol. which previously caused a segmentation fault when an asterisk was used after an async private method declaration.
Fix parsing of invalid \textcolor{red}{asterix} \textcolor{teal}{asterisk} symbol after private field symbol. \textcolor{teal}{which previously caused a segmentation fault when an asterisk was used after an async private method declaration.}


In [4]:
import yaml

msgyaml = '../msg/gemini.yaml'
with open(msgyaml, 'r') as f:
    msgdata = yaml.safe_load(f)

enhance_enabled = []
enhance_score_improve = []
enhance_result_improve = []
enhance_origN = []
enhance2B = []
enhance_N2B = []
enhance_R2B = []
enhance_D2B = []
enhance_N2D = []
enhance_N2R = []

reduce_score_worse = []
reduce_result_worse = []
reduce_origB = []
reduce_B2N = []
reduce_B2R = []
reduce_B2D = []
reduce_origR = []
reduce_R2N = []

best_results = []

for item in msgdata:
  proj, issue, commit = item['proj'], item['issue'], item['commit']
  score_msgonly = item['score_msgonly']
  score_enhance = item.get('score_enhanced', None)
  score_reduce = item.get('score_reduced', None)
  result_msgonly = item['best_result_msgonly']
  result_enhance = item.get('best_result_enhanced', None)
  result_reduce = item.get('best_result_reduced', None)
  best_results.append(result_enhance or result_msgonly)
  if score_enhance is not None:
    enhance_enabled.append(commit)
    if score_enhance > score_msgonly:
      enhance_score_improve.append(commit)
  if result_msgonly != 'B' and result_enhance == 'B':
    enhance2B.append(commit)
  if result_msgonly == 'N':
    enhance_origN.append(commit)
    if result_enhance == 'B':
      enhance_N2B.append(commit)
    if result_enhance == 'D':
      enhance_N2D.append(commit)
    if result_enhance == 'R':
      enhance_N2R.append(commit)
  if result_msgonly == 'R' and result_enhance == 'B':
    enhance_R2B.append(commit)
  if result_msgonly == 'D' and result_enhance == 'B':
    enhance_D2B.append(commit)
  
  if score_reduce is not None and score_reduce < score_msgonly:
    reduce_score_worse.append(commit)
  if result_msgonly == 'B':
    reduce_origB.append(commit)
    if result_reduce == 'N':
      reduce_B2N.append(commit)
    if result_reduce == 'R':
      reduce_B2R.append(commit)
    if result_reduce == 'D':
      reduce_B2D.append(commit)
  if result_msgonly == 'R':
    reduce_origR.append(commit)
    if result_reduce == 'N':
      reduce_R2N.append(commit)

print(f"Enhance improve score: {len(enhance_score_improve)}")
print(f"Enhance enabled: {len(enhance_enabled)}")
print(f"Enhance 2B: {len(enhance2B)}")
print(f"Enhance orig N: {len(enhance_origN)}")
print(f"Enhance N2B: {len(enhance_N2B)}")
print(f"Enhance N2D: {len(enhance_N2D)}")
print(f"Enhance N2R: {len(enhance_N2R)}")
print(f"Enhance R2B: {len(enhance_R2B)}")
print(f"Enhance D2B: {len(enhance_D2B)}")
print(f"Reduce score worse: {len(reduce_score_worse)}")
print(f"Reduce orig B: {len(reduce_origB)}")
print(f"Reduce B2N: {len(reduce_B2N)}, {reduce_B2N}")
print(f"Reduce B2R: {len(reduce_B2R)}")
print(f"Reduce B2D: {len(reduce_B2D)}")
print(f"Reduce orig R: {len(reduce_origR)}")
print(f"Reduce R2N: {len(reduce_R2N)}")
print(f"Number of best results with (enhanced) msg triggering bugs: {len([r for r in best_results if r == 'B'])}")

Enhance improve score: 17
Enhance enabled: 29
Enhance 2B: 7
Enhance orig N: 14
Enhance N2B: 5
Enhance N2D: 2
Enhance N2R: 1
Enhance R2B: 2
Enhance D2B: 0
Reduce score worse: 11
Reduce orig B: 13
Reduce B2N: 1, ['f397a3e']
Reduce B2R: 4
Reduce B2D: 1
Reduce orig R: 9
Reduce R2N: 3
Number of best results with (enhanced) msg triggering bugs: 18


In [5]:
import re
# generate table for reduce score worse with following header
# proj, issue, msg, msg_reduce, score_msgonly, score_reduce, result_msgonly, result_reduce

table_reduce_worse = []
for item in msgdata:
  proj, issue, commit = item['proj'], item['issue'], item['commit']
  score_msgonly = item['score_msgonly']
  score_reduce = item.get('score_reduced', None)
  result_msgonly = item['best_result_msgonly']
  result_reduce = item.get('best_result_reduced', None)
  msg = item['msg']
  msg_reduce = item.get('msg_reduced', None)
  # calculate reduced words
  if msg_reduce:
    words_msg = re.findall(r'\S+', msg)
    words_reduce = re.findall(r'\S+', msg_reduce)
    wc_reduced = len(words_msg) - len(words_reduce)
  if commit in reduce_score_worse:
    table_reduce_worse.append({
      'proj': proj,
      'issue': issue,
      'msg': msg,
      'msg_reduce': msg_reduce,
      'score_msgonly': score_msgonly,
      'score_reduce': score_reduce,
      'result_msgonly': result_msgonly,
      'result_reduce': result_reduce,
      'wc_reduced': wc_reduced,
    })

df_table_reduce_worse = pd.DataFrame(table_reduce_worse)
display(df_table_reduce_worse)
# acount avg reduced words
avg_wc_reduced = df_table_reduce_worse['wc_reduced'].mean()
print(f"Average reduced words: {avg_wc_reduced}")

,proj,issue,msg,msg_reduce,score_msgonly,score_reduce,result_msgonly,result_reduce,wc_reduced
0,mujs,145,Fix js_strtol,Fix strtol,2.4,1.8,B,B,0
1,mujs,166,Issue #166: Use special iterator for string an...,Issue #166: Use special iterator for indices.\...,3.0,1.6,B,B,3
2,libxml2,720,[CVE-2024-34459] Fix buffer overread with `xml...,[CVE-2024-34459] Fix buffer overread with `xml...,1.0,0.9,R,R,1
3,jerryscript,5013,Mark generator as running during yield* (#5014...,Mark generator as running (#5014)\n\nFixes #50...,1.0,0.0,R,N,2
4,jerryscript,5117,Fix arguments object in eval-ed functions in s...,Fix arguments object in functions in static cl...,1.4,1.0,B,R,1
5,z3,6914,fix #6914,#6914,0.2,0.0,R,N,1
6,php-src,16777,Fix GH-16777: Calling the constructor again on...,Fix GH-16777: Calling the constructor again on...,0.6,0.0,R,N,1
7,jq,2976,Merge pull request from GHSA-686w-5m7m-54vc\n\...,Merge pull request from GHSA-686w-5m7m-54vc\n\...,3.0,1.0,B,R,24
8,micropython,13007,py/objslice: Validate that the argument to ind...,py/objslice: Validate that the argument is an ...,0.5,0.0,B,N,2
9,micropython,13041,py/objint: Fix int.to_bytes() buffer size chec...,py/objint: Fix int.to_bytes() buffer size chec...,2.1,2.0,B,D,142


Average reduced words: 17.363636363636363


In [6]:
def openai_count_token(s: str) -> int:
  import tiktoken
  encoding = tiktoken.encoding_for_model("gpt-4o")
  tokens = encoding.encode(s)
  return len(tokens)

# generate table for enhance2B worse with following headr
# proj, issue, msg, msg_enhance, score_msgonly, score_enhance, result_msgonly, result_enhance
table_enhance_2B = []
for item in msgdata:
  proj, issue, commit = item['proj'], item['issue'], item['commit']
  score_msgonly = item['score_msgonly']
  score_enhance = item.get('score_enhanced', None)
  result_msgonly = item['best_result_msgonly']
  result_enhance = item.get('best_result_enhanced', None)
  msg = item['msg']
  msg_enhance = item.get('msg_enhanced', None)
  if msg_enhance:
    words_msg = re.findall(r'\S+', msg)
    words_enhance = re.findall(r'\S+', msg_enhance)
    # wc_added = len(tokens_enhance) - len(tokens_msg)
    wc_added = len(words_enhance) - len(words_msg)
  else:
    wc_added = 0
  if commit in enhance2B:
    table_enhance_2B.append({
      'proj': proj,
      'issue': issue,
      'msg': msg,
      'msg_enhance': msg_enhance,
      'score_msgonly': score_msgonly,
      'score_enhance': score_enhance,
      'result_msgonly': result_msgonly,
      'result_enhance': result_enhance,
      'wc_added': wc_added,
    })
df_table_enhance_2B = pd.DataFrame(table_enhance_2B)
display(df_table_enhance_2B)

,proj,issue,msg,msg_enhance,score_msgonly,score_enhance,result_msgonly,result_enhance,wc_added
0,mujs,65,Fix issue #65: Uninitialized name in Function....,Fix issue #65: Uninitialized `name` and `lengt...,1.0,3.0,R,B,25
1,mujs,141,Issue #141: Add missing end-of-string checks i...,Issue #141: Add missing end-of-string checks i...,0.0,3.0,N,B,22
2,poppler,1289,pdfunite: Don't crash in broken documents,pdfunite: Don't crash when merging documents c...,0.0,2.1,N,B,11
3,jerryscript,5153,Fix parsing of invalid asterix symbol after pr...,Fix parsing of invalid asterisk symbol after p...,0.0,1.2,N,B,16
4,z3,7252,fix #7252\n\nSigned-off-by: Nikolaj Bjorner <n...,fix #7252 segmentation fault in nested re.loop...,0.0,1.6,N,B,19
5,micropython,17841,extmod/vfs_reader: Check that open() resulted ...,extmod/vfs_reader: Check that open() resulted ...,1.0,1.2,R,B,-27
6,micropython,17847,py/objringio: Detect incorrect constructor cal...,py/objringio: Detect incorrect constructor cal...,0.0,2.4,N,B,35


In [7]:
def openai_count_token(s: str) -> int:
  import tiktoken
  encoding = tiktoken.encoding_for_model("gpt-4o-2024-08-06")
  tokens = encoding.encode(s)
  return len(tokens)

# generate table for enhance2B worse with following headr
# proj, issue, msg, msg_enhance, score_msgonly, score_enhance, result_msgonly, result_enhance
table_msg = []
for item in msgdata:
  proj, issue, commit = item['proj'], item['issue'], item['commit']
  score_msgonly = item['score_msgonly']
  score_enhance = item.get('score_enhanced', None)
  score_reduce = item.get('score_reduced', None)
  result_msgonly = item['best_result_msgonly']
  result_enhance = item.get('best_result_enhanced', None)
  result_reduce = item.get('best_result_reduced', None)
  msg = item['msg']
  msg_enhance = item.get('msg_enhanced', None)
  msg_reduce = item.get('msg_reduced', None)
  wc_added = wc_reduced = 0
  if msg_enhance:
    # wc_added = openai_count_token(msg_enhance) - openai_count_token(msg)
    words_msg = re.findall(r'\S+', msg)
    words_enhance = re.findall(r'\S+', msg_enhance)
    wc_added = len(words_enhance) - len(words_msg)
  if msg_reduce:
    # wc_reduced = openai_count_token(msg) - openai_count_token(msg_reduce)
    words_msg = re.findall(r'\S+', msg)
    words_reduce = re.findall(r'\S+', msg_reduce)
    wc_reduced = len(words_msg) - len(words_reduce)
  table_msg.append({
    'proj': proj,
    'issue': issue,
    'msg': msg,
    'msg_enhance': msg_enhance,
    'msg_reduce': msg_reduce,
    'score_msgonly': score_msgonly,
    'score_enhance': score_enhance,
    'score_reduce': score_reduce,
    'result_msgonly': result_msgonly,
    'result_enhance': result_enhance,
    'result_reduce': result_reduce,
    'wc_added': wc_added,
    'wc_reduced': wc_reduced,
  })
df_table_msg = pd.DataFrame(table_msg)
display(df_table_msg)

# show wc_added avg, only for rows with msg_enhance not None
df_table_enhance = df_table_msg[df_table_msg['msg_enhance'].notna()]
avg_wc_added = df_table_enhance['wc_added'].mean()
# calculate percentage of added words over original msg
print(f"Average added words: {avg_wc_added}")
print(df_table_enhance['wc_added'].describe())

df_table_reduce = df_table_msg[df_table_msg['msg_reduce'].notna()]
df_table_reduce = df_table_reduce[df_table_reduce['score_reduce'] < df_table_reduce['score_msgonly']]
display(df_table_reduce)
avg_wc_reduced = df_table_reduce['wc_reduced'].mean()
print(f"Average reduced words: {avg_wc_reduced}")
print(df_table_reduce['wc_reduced'].describe())



,proj,issue,msg,msg_enhance,msg_reduce,score_msgonly,score_enhance,score_reduce,result_msgonly,result_enhance,result_reduce,wc_added,wc_reduced
0,mujs,65,Fix issue #65: Uninitialized name in Function....,Fix issue #65: Uninitialized `name` and `lengt...,Fix issue #65: Uninitialized name in Function,1.0,3.0,1.0,R,B,R,25,1
1,mujs,141,Issue #141: Add missing end-of-string checks i...,Issue #141: Add missing end-of-string checks i...,NaN,0.0,3.0,NaN,N,B,NaN,22,0
2,mujs,145,Fix js_strtol,Fix js_strtol when the input string contains c...,Fix strtol,2.4,3.0,1.8,B,B,B,21,0
3,mujs,166,Issue #166: Use special iterator for string an...,NaN,Issue #166: Use special iterator for indices.\...,3.0,NaN,1.6,B,NaN,B,0,3
4,libxml2,535,parser: Fix old SAX1 parser with custom callba...,NaN,parser: Fix old SAX1 parser with custom callba...,3.0,NaN,3.0,B,NaN,B,0,47
5,libxml2,550,xpath: Ignore entity ref nodes when computing ...,xpath: Ignore entity ref nodes when computing ...,NaN,0.0,0.0,NaN,N,N,NaN,-20,0
6,libxml2,553,malloc-fail: Handle malloc failures in xmlAddE...,NaN,malloc-fail: Handle malloc failures\n\nAvoid m...,3.0,NaN,3.0,B,NaN,B,0,2
7,libxml2,720,[CVE-2024-34459] Fix buffer overread with `xml...,[CVE-2024-34459] Fix buffer overread in `xmlli...,[CVE-2024-34459] Fix buffer overread with `xml...,1.0,1.0,0.9,R,R,R,23,1
8,libxml2,841,xmllint: Fix UAF with --push --repeat\n\nShort...,NaN,xmllint: Fix UAF\n\nShort-lived regression. Fi...,3.0,NaN,3.0,B,NaN,B,0,3
9,poppler,1282,pdfunite: Fix crash on broken files,pdfunite: Fix crash on broken files when `PDFD...,pdfunite: Fix crash,1.0,0.9,1.0,R,R,R,12,3


Average added words: 17.482758620689655
count    29.000000
mean     17.482759
std      14.989569
min     -27.000000
25%      12.000000
50%      19.000000
75%      25.000000
max      43.000000
Name: wc_added, dtype: float64


,proj,issue,msg,msg_enhance,msg_reduce,score_msgonly,score_enhance,score_reduce,result_msgonly,result_enhance,result_reduce,wc_added,wc_reduced
2,mujs,145,Fix js_strtol,Fix js_strtol when the input string contains c...,Fix strtol,2.4,3.0,1.8,B,B,B,21,0
3,mujs,166,Issue #166: Use special iterator for string an...,NaN,Issue #166: Use special iterator for indices.\...,3.0,NaN,1.6,B,NaN,B,0,3
7,libxml2,720,[CVE-2024-34459] Fix buffer overread with `xml...,[CVE-2024-34459] Fix buffer overread in `xmlli...,[CVE-2024-34459] Fix buffer overread with `xml...,1.0,1.0,0.9,R,R,R,23,1
14,jerryscript,5013,Mark generator as running during yield* (#5014...,Mark generator as running during yield* when a...,Mark generator as running (#5014)\n\nFixes #50...,1.0,1.0,0.0,R,R,N,31,2
16,jerryscript,5117,Fix arguments object in eval-ed functions in s...,Fix arguments object in eval-ed functions in s...,Fix arguments object in functions in static cl...,1.4,2.8,1.0,B,B,R,31,1
19,z3,6914,fix #6914,fix #6914: heap-use-after-free with str.to_int...,#6914,0.2,0.7,0.0,R,R,N,12,1
22,php-src,16777,Fix GH-16777: Calling the constructor again on...,Fix GH-16777: Calling the constructor again on...,Fix GH-16777: Calling the constructor again on...,0.6,0.7,0.0,R,R,N,36,1
27,jq,2976,Merge pull request from GHSA-686w-5m7m-54vc\n\...,NaN,Merge pull request from GHSA-686w-5m7m-54vc\n\...,3.0,NaN,1.0,B,NaN,R,0,24
30,micropython,13007,py/objslice: Validate that the argument to ind...,py/objslice: Validate that the argument to `in...,py/objslice: Validate that the argument is an ...,0.5,1.0,0.0,B,D,N,25,2
31,micropython,13041,py/objint: Fix int.to_bytes() buffer size chec...,py/objint: Fix int.to_bytes() buffer size chec...,py/objint: Fix int.to_bytes() buffer size chec...,2.1,2.0,2.0,B,D,D,43,142


Average reduced words: 17.363636363636363
count     11.000000
mean      17.363636
std       42.000649
min        0.000000
25%        1.000000
50%        2.000000
75%        8.500000
max      142.000000
Name: wc_reduced, dtype: float64


In [8]:
enhanced_examples = [7252, 65, 5153]

for issue in enhanced_examples:
  msg = df_table_enhance_2B[df_table_enhance_2B['issue'] == issue]['msg'].values[0]
  msg = msg.splitlines()[0]
  msg_enhance = df_table_enhance_2B[df_table_enhance_2B['issue'] == issue]['msg_enhance'].values[0]
  msg_enhance = msg_enhance.splitlines()[0]
  msg_diff = word_diff(msg, msg_enhance, latex=True)
  print(f"Issue {issue} diff:")
  print(msg_diff)

reduced_examples = [5117]
for issue in reduced_examples:
  msg = df_table_reduce_worse[df_table_reduce_worse['issue'] == issue]['msg'].values[0]
  msg = msg.splitlines()[0]
  msg_reduce = df_table_reduce_worse[df_table_reduce_worse['issue'] == issue]['msg_reduce'].values[0]
  msg_reduce = msg_reduce.splitlines()[0]
  msg_diff = word_diff(msg, msg_reduce, latex=True)
  print(f"Issue {issue} diff:")
  print(msg_diff)


Issue 7252 diff:
fix #7252 \textcolor{teal}{segmentation fault in nested re.loop formula processing when handling specific boundary values for nested re.loop expressions with same bounds of 0 and 1}
Issue 65 diff:
Fix issue #65: Uninitialized \textcolor{red}{name} \textcolor{teal}{`name` and `length` properties} in \textcolor{red}{Function.prototype function.} \textcolor{teal}{`Function.prototype` causing a null pointer dereference when `Function.prototype` is converted to a string, specifically observed when `Array.prototype.splice` is called with `Function.prototype` as an argument.}
Issue 5153 diff:
Fix parsing of invalid \textcolor{red}{asterix} \textcolor{teal}{asterisk} symbol after private field symbol \textcolor{red}{(#5155)} \textcolor{teal}{which previously caused a segmentation fault when an asterisk was used after an async private method declaration.}
Issue 5117 diff:
Fix arguments object in \textcolor{red}{eval-ed} functions in static class initializers (#5140)
